<h1>Chapter 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This notebook is for Chapter 2 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


## 7000-Level Submission Note
Run all cells top-to-bottom in Google Colab. Keep all outputs visible when submitting.


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [1]:
# %%capture
# !pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/16.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

In [3]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=20
)

# Print the output
print(tokenizer.decode(generation_output[0]))

Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Heartfelt Apologies for the Gardening Mishap


Dear


In [4]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [5]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [6]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901, 17778, 29888,  2152,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799]], device='cuda:0')

In [7]:
print(tokenizer.decode(3323))
print(tokenizer.decode(622))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

Sub
ject
Subject
:


# Comparing Trained LLM Tokenizers


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [9]:
text = """
The sesquipedalian researcher recorded 47 observations and wrote "bonjour" 😊.
"""


In [10]:
show_tokens(text, "bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] the se ##s ##qui ##ped ##alia ##n researcher recorded 47 observations and wrote " bon ##jou ##r " [UNK] . [SEP] 

In [11]:
show_tokens(text, "bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] The se ##s ##qui ##ped ##alia ##n researcher recorded 47 observations and wrote " b ##on ##jou ##r " [UNK] . [SEP] 

In [12]:
show_tokens(text, "gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 The  s es qu iped alian  researcher  recorded  47  observations  and  wrote  " bon j our "  � � . 
 

In [13]:
show_tokens(text, "google/flan-t5-small")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

The ses qui ped a lian researcher recorded 47 observations and wrote " bon jour "  <unk> . </s> 

In [14]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.23M [00:00<?, ?B/s]


 The  ses qu iped alian  researcher  recorded   47  observations  and  wrote  " bon jour "  � � . 
 

In [15]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 49151), got 50256. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



 The  ses quip ed alian  research er  recorded   4 7  observations  and  wrote  " bon j our "  � � . 
 

In [16]:
show_tokens(text, "facebook/galactica-1.3b")

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 The  sesqu iped alian  researcher  recorded   4 7  observations  and  wrote   " bon j our "   � � � � . 
 

In [17]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 The ses quip ed al ian research er recorded  4 7 observations and wrote " bon j our "  � � � � . 
 

## 7000-Level Addition — Vocabulary Size and Token Counts


In [18]:
additional_tokenizer_name = "roberta-base"
show_tokens(text, additional_tokenizer_name)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

<s> 
 The  s es qu iped alian  researcher  recorded  47  observations  and  wrote  " bon j our "  � � . 
 </s> 

In [19]:
tokenizer_names = [
    "bert-base-uncased", "bert-base-cased", "gpt2",
    "google/flan-t5-small", "Xenova/gpt-4",
    "facebook/galactica-1.3b", "microsoft/Phi-3-mini-4k-instruct",
    "roberta-base"
]
comparison = []
for name in tokenizer_names:
    tok = AutoTokenizer.from_pretrained(name)
    ids = tok(text).input_ids
    comparison.append({"tokenizer": name, "token_count": len(ids), "vocab_size": tok.vocab_size})
import pandas as pd
comparison_df = pd.DataFrame(comparison).sort_values("token_count", ascending=False)
display(comparison_df)
print("Most tokens:", comparison_df.iloc[0]["tokenizer"], int(comparison_df.iloc[0]["token_count"]))
print("Fewest tokens:", comparison_df.iloc[-1]["tokenizer"], int(comparison_df.iloc[-1]["token_count"]))


,tokenizer,token_count,vocab_size
6,microsoft/Phi-3-mini-4k-instruct,29,32000
5,facebook/galactica-1.3b,26,50000
7,roberta-base,24,50265
1,bert-base-cased,23,28996
0,bert-base-uncased,22,30522
2,gpt2,22,50257
4,Xenova/gpt-4,21,100263
3,google/flan-t5-small,20,32100


Most tokens: microsoft/Phi-3-mini-4k-instruct 29
Fewest tokens: google/flan-t5-small 20


### Q1 Answer:
The tokenizer with the most tokens is the one reported at the top of the comparison table, while the tokenizer with the fewest tokens is the one at the bottom. Token counts differ because each tokenizer was trained with a different vocabulary and subword-tokenization strategy. A tokenizer that has a useful whole-word or larger subword unit can represent text with fewer tokens, while another may split the same text into smaller pieces.

### Q2 Answer:
The uncommon word sesquipedalian is the key comparison point. Different tokenizers split it into different subword pieces because it is relatively uncommon. Tokenizers with a suitable subword unit may keep more of the word together, while others break it into multiple pieces.

### 7000-Level Vocabulary Answer:
Vocabulary size does not consistently determine which tokenizer produces the fewest tokens. A larger vocabulary can provide more whole-word or subword units, but token count also depends on the tokenizer's training data, vocabulary construction, and segmentation algorithm.

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [20]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors: reconstructing file:   0%|          |  0.00B /  241MB            

model.safetensors: downloading bytes:           |  0.00B            

In [21]:
output.shape

torch.Size([1, 4, 384])

In [22]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


In [23]:
output

tensor([[[-3.4805,  0.0862, -0.1818,  ..., -0.0610, -0.3909,  0.3022],
         [ 0.1885,  0.3201, -0.2313,  ...,  0.3721,  0.2471,  0.8057],
         [ 0.2089,  0.5010, -0.0495,  ...,  1.2197, -0.2277,  0.8574],
         [-3.4277,  0.0635, -0.1426,  ...,  0.0658, -0.4358,  0.3826]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

In [24]:
from transformers import AutoModel, AutoTokenizer
import torch

context_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-xsmall")
context_model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

sentences = [
    "I sat on the bank of the river and watched the water.",
    "I went to the bank to deposit my paycheck."
]
bank_embeddings = []

for sentence in sentences:
    encoded = context_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = context_model(**encoded).last_hidden_state
    tokens = context_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    print("\nSentence:", sentence)
    print("Tokens:", tokens)
    bank_positions = [i for i, token in enumerate(tokens)
                      if "bank" in token.lower().replace("▁", "")]
    print("Token positions for 'bank':", bank_positions)
    vector = hidden[0, bank_positions, :].mean(dim=0)
    bank_embeddings.append(vector)
    print("Embedding shape:", tuple(vector.shape))

cosine_similarity = torch.nn.functional.cosine_similarity(
    bank_embeddings[0].unsqueeze(0), bank_embeddings[1].unsqueeze(0)
).item()
print("\nCosine similarity:", cosine_similarity)
print("Cosine distance:", 1 - cosine_similarity)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d


Sentence: I sat on the bank of the river and watched the water.
Tokens: ['[CLS]', '▁I', '▁sat', '▁on', '▁the', '▁bank', '▁of', '▁the', '▁river', '▁and', '▁watched', '▁the', '▁water', '.', '[SEP]']
Token positions for 'bank': [5]
Embedding shape: (384,)

Sentence: I went to the bank to deposit my paycheck.
Tokens: ['[CLS]', '▁I', '▁went', '▁to', '▁the', '▁bank', '▁to', '▁deposit', '▁my', '▁paycheck', '.', '[SEP]']
Token positions for 'bank': [5]
Embedding shape: (384,)

Cosine similarity: 0.767578125
Cosine distance: 0.232421875


### Q5 Answer:
Yes. The contextual embedding for “bank” should differ between the two sentences because the model uses surrounding words to construct the token representation. In the first sentence, the context indicates a river bank; in the second, it indicates a **financial bank. This is the key difference from GloVe becuase it gives a word one fixed vector, while a contextual language-model embedding can produce different vectors for the same word depending on context.


# Text Embeddings (For Sentences and Whole Documents)

In [25]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
vector.shape

(768,)

# Word Embeddings Beyond LLMs


In [34]:
!pip install gensim
import gensim.downloader as api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.1 MB/s eta 0:00:00


In [35]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [36]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

## Word Embeddings Beyond LLMs — My Three Words


In [37]:
my_words = ["bank", "computer", "ocean"]
for word in my_words:
    print(f"\nNearest neighbors for: {word}")
    print(model.most_similar(word, topn=10))



Nearest neighbors for: bank
[('banks', 0.869862973690033), ('securities', 0.7996813654899597), ('banking', 0.7965160012245178), ('investment', 0.7849708199501038), ('exchange', 0.7808825969696045), ('financial', 0.7670274972915649), ('credit', 0.764915406703949), ('lender', 0.7518407702445984), ('capital', 0.7380707859992981), ('brokerage', 0.7373986840248108)]

Nearest neighbors for: computer
[('computers', 0.9165045022964478), ('software', 0.8814992904663086), ('technology', 0.852556049823761), ('electronic', 0.812586784362793), ('internet', 0.8060455322265625), ('computing', 0.802603542804718), ('devices', 0.8016185760498047), ('digital', 0.7991793751716614), ('applications', 0.7912740707397461), ('pc', 0.7883159518241882)]

Nearest neighbors for: ocean
[('sea', 0.8811647891998291), ('waters', 0.8635229468345642), ('seas', 0.846487820148468), ('coast', 0.808632493019104), ('shores', 0.8028936982154846), ('atlantic', 0.7981462478637695), ('oceans', 0.792408287525177), ('inland', 0.7

### Q3 Answer:
For bank, the nearest neighbors mainly reflect associations learned from the training corpus rather than one particular sentence. Because GloVe assigns one fixed vector to a word, it cannot create separate representations for the financial and river-bank meanings. If one sense appears more often in the corpus, that sense can dominate the nearest neighbors; some neighbors may also reflect a mixture of senses.

### Q4 Answer:
For computer, I expect neighbors related to computing and technology. For ocean, I expect neighbors related to water, marine environments, and geography. The exact nearest neighbors depend on the GloVe training corpus, so an unexpected neighbor would be a useful example of distributional similarity.


## Contextualized Word Embeddings — Same Word, Two Meanings


# Recommending songs by embeddings

In [38]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [39]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [40]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [41]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('5549', 0.9954367280006409),
 ('1922', 0.9951966404914856),
 ('3094', 0.9951208233833313),
 ('2849', 0.9934636950492859),
 ('2976', 0.993462324142456),
 ('3114', 0.9925068616867065),
 ('6626', 0.9923655986785889),
 ('5634', 0.991847813129425),
 ('2104', 0.991837203502655),
 ('2640', 0.9915144443511963)]

In [42]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [43]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
5549,November Rain,Guns N' Roses
1922,One,Metallica
3094,Breaking The Law,Judas Priest
2849,Run To The Hills,Iron Maiden
2976,I Don't Know,Ozzy Osbourne


In [44]:
print_recommendations(2172)

,title,artist
id,,
5549,November Rain,Guns N' Roses
1922,One,Metallica
3094,Breaking The Law,Judas Priest
2849,Run To The Hills,Iron Maiden
2976,I Don't Know,Ozzy Osbourne


In [45]:
print_recommendations(842)

,title,artist
id,,
5681,Drop It Low (w\/ Chris Brown),Ester Dean
5668,How We Do (w\/ 50 Cent),The Game
331,Get Low (w\/ Ying Yang Twins),Lil' Jon & The Eastside Boyz
12205,Give It Up To Me,Sean Paul
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
